# Track 1: ЭЭГ → изображение

По окну ЭЭГ (−0.2…0.8 с вокруг показа картинки) предсказывается эмбеддинг картинки DINOv2-giant $y\in\mathbb R^{1536}$.
Метрика — top-5: предсказание $p$ ранжируется по $\cos(p, y_j)$ среди эмбеддингов тестовых картинок; в закрытом этапе предсказания
всех повторов картинки у одного человека сначала складываются. Закрытый этап — новые люди и новые картинки.

**Энкодер** (`nets.py`). Каналы задаются не номерами, а 3D-координатами электродов: виртуальный канал $v$ смешивает входы
с весами $\mathrm{softmax}_c\,(q_v^\top\varphi(\mathrm{pos}_c))$, где $\varphi$ — признаки Фурье координат. Поэтому одна модель читает любой монтаж,
отсутствующие каналы просто не участвуют; при обучении из каждого окна выбрасывается случайная область скальпа.
Дальше — остаточные свёртки с расширением по времени, пулинг по времени с вниманием (несколько голов, у каждой свой профиль латентности)
и MLP: $p=\mu+s\,u$. От человека ничего не зависит.

**Обучение** (`encoder.py`) в геометрии метрики: InfoNCE по $\cos(p, y)$ с исходными эмбеддингами, без центрирования.
В батче $G$ картинок, у каждой $V$ видов; вид — $k$ повторов картинки у одного человека, его предсказание — среднее их предсказаний,
как в усреднённой метрике. Виды одной картинки от разных людей — все позитивы.

**Валидация** — репетиция закрытого этапа: несколько людей целиком не участвуют в обучении и оцениваются на тестовых картинках.

**Приор** (`prior.py`, необязательный): выпрямленный поток от предсказания к настоящему эмбеддингу. Учится на половине отложенных людей
на обучающих картинках, число шагов (0 — без приора) выбирается на другой половине на тестовых картинках: для приора они тоже новые люди.

Отправка одной командой из папки `track1_image`: `python -m image_decoding.make_submission --study xu2025alljoined`.

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, torch
from image_decoding import data
from image_decoding.encoder import held_out_subjects, split, target_space, train_encoder
from image_decoding.evaluation import evaluate, predict, REFERENCE
from image_decoding.nets import ImageDecoder
from image_decoding.prior import fit_prior, score_steps

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SYNTHETIC = False  # True: маленькие фейковые данные, чтобы проверить код без скачивания
packed = (data.synthetic() if SYNTHETIC else data.load("xu2025alljoined", "../data")).to(DEVICE)
NET = dict(width=64, hidden=256, n_virtual=32, images_per_batch=64) if SYNTHETIC else {}
EPOCHS, SEEDS = (15, 2) if SYNTHETIC else (30, 3)
held = held_out_subjects(packed, 2)
train_idx, val_idx = split(packed, held)
packed.summary()

## Энкодеры
Каждый сид — отдельный энкодер, лучшая эпоха — по top-5 с усреднением повторов у отложенных людей.

In [ ]:
encoders, histories = [], []
for seed in range(SEEDS):
    net, history = train_encoder(packed, held, epochs=EPOCHS, device=DEVICE, seed=seed, **NET)
    encoders.append(net); histories.append(pd.DataFrame(history).set_index("epoch"))
pd.concat({f"сид {i}": h["top5_subject_agg"] for i, h in enumerate(histories)}, axis=1).plot(
    title="отложенные люди, top-5 с усреднением, %");

Необязательно: сравнение добавок к потере на одном сиде — мягкие цели и MSE. Если какая-то лучше, передайте её в `make_submission` (`--soft`, `--mse-weight`).

In [ ]:
ABLATION = False
if ABLATION:
    variants = {"InfoNCE": {}, "+ мягкие цели": dict(soft=0.1), "+ MSE": dict(mse_weight=0.5)}
    rows = {name: pd.DataFrame(train_encoder(packed, held, epochs=EPOCHS, device=DEVICE, **NET, **kw)[1])["top5_subject_agg"].max()
            for name, kw in variants.items()}
    print(pd.Series(rows, name="лучший top-5 с усреднением, %").round(2))

## Ансамбль
Предсказания членов делятся на их типичную норму и усредняются.

In [ ]:
mu, s, radius = target_space(packed)
model = ImageDecoder(encoders, mu, s, radius, packed.positions, packed.ch_names).to(DEVICE)
for m, enc in enumerate(encoders):
    single = ImageDecoder([enc], mu, s, radius, packed.positions, packed.ch_names).to(DEVICE)
    model.scales[m] = float(np.linalg.norm(predict(single, packed, val_idx), axis=1).mean())
evaluate(model, packed, val_idx)

## Приор
Учится на окнах, которых энкодеры не видели: отложенные люди, обучающие картинки.

In [ ]:
learn, judge = held[: len(held) // 2], held[len(held) // 2:]
fit_idx = np.flatnonzero(np.isin(packed.subject, learn) & ~packed.test)
judge_idx = np.flatnonzero(np.isin(packed.subject, judge) & packed.test)
monitor = (predict(model, packed, judge_idx), packed.image[judge_idx], packed.subject[judge_idx], packed.images)
history = fit_prior(model, predict(model, packed, fit_idx), packed.images[packed.image[fit_idx]],
                    epochs=60 if SYNTHETIC else 100, monitor=monitor)
steps = {n: score_steps(model, *monitor, n_steps=n)["top5_subject_agg"] for n in [0, 1, 2, 5, 10, 20]}
model.n_steps = max(steps, key=steps.get)
if model.n_steps == 0:
    model.prior = None
pd.Series(steps).plot(marker="o", xlabel="шаги Эйлера (0 — без приора)", title="вторая половина отложенных, top-5 с усреднением, %")
print("n_steps =", model.n_steps)

## Тестовые картинки
`top5` — каждое окно отдельно (warm-up), `top5_subject_agg` — повторы усреднены внутри человека (закрытый этап).

In [ ]:
seen = np.flatnonzero(packed.test & ~np.isin(packed.subject, held))
print("ориентиры на THINGS-EEG2, top-5 %:", REFERENCE)
pd.DataFrame({"новые люди (как закрытый этап)": evaluate(model, packed, val_idx),
              "люди из обучения": evaluate(model, packed, seen)}).T.round(2)

## ZIP для Codabench

In [ ]:
from image_decoding.export import check_zip, save_submission
check_zip(save_submission(model, "../submissions/spatial_conv"))